In [6]:
import pyodbc
 
def create_users_table():
    conn = pyodbc.connect(
        "Driver={ODBC Driver 17 for SQL Server};"
        "Server=tcp:az26d1-rsl-sqldb-svr01.database.windows.net,1433;"
        "Database=az26d1-rsl-sqldbsvr01;"
        "Authentication=ActiveDirectoryInteractive;"
        "Encrypt=yes;TrustServerCertificate=no;"
        "Connection Timeout=60;"
    )
    cursor = conn.cursor()
 
    # Create rsl.users table
    cursor.execute("""
        IF NOT EXISTS (
            SELECT * FROM sys.tables t
            JOIN sys.schemas s ON t.schema_id = s.schema_id
            WHERE s.name = 'rsl' AND t.name = 'users_new'
        )
        CREATE TABLE rsl.users_new (
            id              INT IDENTITY(1,1) PRIMARY KEY,
            email           NVARCHAR(255) NOT NULL UNIQUE,
            password_hash   NVARCHAR(512) NOT NULL,
            salt            NVARCHAR(128) NOT NULL,
            full_name       NVARCHAR(255) NULL,
            status          NVARCHAR(20) NOT NULL DEFAULT 'pending',
            role            NVARCHAR(20) NOT NULL DEFAULT 'user',
            registered_at   DATETIME2 NOT NULL DEFAULT GETUTCDATE(),
            approved_by     NVARCHAR(255) NULL,
            approved_at     DATETIME2 NULL,
            last_login_at   DATETIME2 NULL,
            CONSTRAINT chk_status CHECK (status IN ('pending', 'approved', 'rejected')),
            CONSTRAINT chk_role CHECK (role IN ('user', 'admin'))
        );
    """)
    print("OK: rsl.users table created (or already exists)")
 
    # Create index on status for quick admin queries
    cursor.execute("""
        IF NOT EXISTS (
            SELECT * FROM sys.indexes
            WHERE object_id = OBJECT_ID('rsl.users_new') AND name = 'IX_users_status'
        )
        CREATE INDEX IX_users_status ON rsl.users_new (status);
    """)
    print("OK: IX_users_status index created")
 
    conn.commit()
    conn.close()
    print("\n✅ Migration complete. rsl.users_new table is ready.")
 
 
if __name__ == "__main__":
    create_users_table()

ProgrammingError: ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open server 'az26d1-rsl-sqldb-svr01' requested by the login. Client with IP address '58.84.61.100' is not allowed to access the server.  To enable access, use the Azure Management Portal or run sp_set_firewall_rule on the master database to create a firewall rule for this IP address or address range.  It may take up to five minutes for this change to take effect. (40615) (SQLDriverConnect); [42000] [Microsoft][ODBC Driver 17 for SQL Server]Invalid connection string attribute (0); [42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open server 'az26d1-rsl-sqldb-svr01' requested by the login. Client with IP address '58.84.61.100' is not allowed to access the server.  To enable access, use the Azure Management Portal or run sp_set_firewall_rule on the master database to create a firewall rule for this IP address or address range.  It may take up to five minutes for this chang

In [7]:
import pyodbc
import sus_app

# sus_app.get_connection() uses Managed Identity (works only inside Azure).
# For local runs, point it at interactive AAD auth so run_query() works here.
def _local_connection():
    return pyodbc.connect(
        f"Driver={{{sus_app.SQL_DRIVER}}};"
        f"Server=tcp:{sus_app.SQL_SERVER},1433;"
        f"Database={sus_app.SQL_DATABASE};"
        "Authentication=ActiveDirectoryInteractive;"
        "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60;"
    )

sus_app.get_connection = _local_connection
sus_app._reset_connection()  # drop any pooled MSI connection

# Run SQL through sus_app's helper
df = sus_app.run_query("SELECT TOP 5 name AS table_name FROM sys.tables ORDER BY name;")
print(df)


RuntimeError: Directory 'c:\Users\320241208\Documents\STET Systeur App\static' does not exist

In [9]:
import pyodbc

def get_connection():
    """Connect to Azure SQL (local use → interactive AAD login)."""
    return pyodbc.connect(
        "Driver={ODBC Driver 17 for SQL Server};"
        "Server=tcp:az26d1-rsl-sqldb-svr01.database.windows.net,1433;"
        "Database=az26d1-rsl-sqldbsvr01;"
        "Authentication=ActiveDirectoryInteractive;"
        "Encrypt=yes;TrustServerCertificate=no;"
        "Connection Timeout=60;"
    )

# Test it
conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT DB_NAME(), SUSER_SNAME();")
db, login = cursor.fetchone()
print(f"✅ Connected to {db} as {login}")
cursor.close()
conn.close()


ProgrammingError: ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open server 'az26d1-rsl-sqldb-svr01' requested by the login. Client with IP address '58.84.61.100' is not allowed to access the server.  To enable access, use the Azure Management Portal or run sp_set_firewall_rule on the master database to create a firewall rule for this IP address or address range.  It may take up to five minutes for this change to take effect. (40615) (SQLDriverConnect); [42000] [Microsoft][ODBC Driver 17 for SQL Server]Invalid connection string attribute (0); [42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open server 'az26d1-rsl-sqldb-svr01' requested by the login. Client with IP address '58.84.61.100' is not allowed to access the server.  To enable access, use the Azure Management Portal or run sp_set_firewall_rule on the master database to create a firewall rule for this IP address or address range.  It may take up to five minutes for this chang